# DEDS — ML Results Review

This notebook reads one frozen Stage 06 result bundle. It does not run ingestion, training, replay, prediction, or hypothesis tests. The figures and tables are for review and communication.

**Research target:** forward realized volatility over the next five XNYS trading sessions, annualized decimal. This is a volatility forecast, not a return forecast.

**Result status:** SOURCE VERIFIED → REVIEW READY after this notebook and the report pack pass the recorded checks. REPORT APPROVED requires the project owner to review and approve the interpretation.

## 00. Context and experiment identity

The question is whether the approved models forecast the next five-session realized volatility for SPY, IEF, and GLD, and how their results compare under the fixed Train / Validation / Test contract.

The comparison and hypothesis outputs are read directly from Stage 06. Macro metrics give each ETF equal weight. Portfolio weights of 50/30/20 are used only for the portfolio-risk diagnostic and do not weight the model comparison.

In [ ]:
from pathlib import Path
import base64
from html import escape
import hashlib
import json
import os

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

EXPECTED_EVALUATION_ID = "82572c8417041976e1ce540d921e29a5f28c1d5d2a2d953a74c03582e7ca9be2"
EXPECTED_REPLAY_ID = "3cbf81eb9242d8894d2f67f39065a17ff1ef55a2e098773aead915949ab11b12"
MODEL_ORDER = [
    "CONSTANT_BASELINE",
    "ROLLING_20D_BASELINE",
    "STATIC_SGD",
    "ONLINE_SGD",
]
MODEL_LABELS = {
    "CONSTANT_BASELINE": "B0",
    "ROLLING_20D_BASELINE": "B1",
    "STATIC_SGD": "B2",
    "ONLINE_SGD": "B3",
}
TICKERS = ["SPY", "IEF", "GLD"]


def locate_bundle() -> Path:
    configured = os.environ.get("DEDS_RESULTS_DIR")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if candidate.is_dir():
            return candidate
        raise FileNotFoundError(f"DEDS_RESULTS_DIR does not exist: {candidate}")

    start = Path.cwd().resolve()
    candidates = [start / "approved_results"]
    for parent in (start, *start.parents):
        candidates.extend(
            [
                parent / "approved_results",
                parent / "dist" / "ml-results-handoff-82572c84" / "approved_results",
            ]
        )
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "execution_manifest.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find approved_results. Set DEDS_RESULTS_DIR to that folder."
    )


BUNDLE_DIR = locate_bundle()
BUNDLE_ROOT = BUNDLE_DIR.parent
FIGURE_DIR = None
if (BUNDLE_ROOT / "START_HERE.md").is_file():
    FIGURE_DIR = BUNDLE_ROOT / "notebook_output" / "figures"
else:
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (parent / ".git").exists():
            FIGURE_DIR = parent / "docs" / "ml-results" / "figures"
            break
if FIGURE_DIR is None:
    FIGURE_DIR = BUNDLE_ROOT / "notebook_output" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def display_figure(figure_path: Path, caption: str) -> None:
    encoded = base64.b64encode(figure_path.read_bytes()).decode("ascii")
    safe_caption = escape(caption, quote=True)
    display(HTML(
        f'<figure><img src="data:image/png;base64,{encoded}" alt="{safe_caption}">'
        f'<figcaption>{safe_caption}</figcaption></figure>'
    ))

print("Results directory: approved_results")
print("Figure output: repository docs/ml-results/figures or portable bundle notebook_output/figures")

## 01. Load and validate

Before analysis, this section confirms the bundle identity, exported-file checksums, required columns, row counts, model/ticker set, and split dates. If a critical file or identity is inconsistent, execution stops here.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


identity_path = BUNDLE_ROOT / "source_identity.json"
identity = json.loads(identity_path.read_text(encoding="utf-8"))
manifest = json.loads((BUNDLE_DIR / "execution_manifest.json").read_text(encoding="utf-8"))

assert identity["evaluation_id"] == EXPECTED_EVALUATION_ID
assert identity["final_replay_experiment_id"] == EXPECTED_REPLAY_ID
assert identity["source_verification_status"] == "SOURCE VERIFIED"
assert manifest["evaluation_id"] == identity["evaluation_id"]
assert manifest["final_replay_experiment_id"] == identity["final_replay_experiment_id"]
assert manifest["status"] == "COMPLETED"

for name, expected_hash in identity["exported_file_sha256"].items():
    path = BUNDLE_DIR / name
    assert path.is_file(), f"Missing result artifact: {name}"
    assert sha256_file(path) == expected_hash, f"Export checksum mismatch: {name}"

checksum_file = BUNDLE_ROOT / "CHECKSUMS.sha256"
checksum_entries = {}
for line in checksum_file.read_text(encoding="utf-8").splitlines():
    digest, relative_path = line.split("  ", 1)
    checksum_entries[relative_path] = digest
for relative_path, digest in checksum_entries.items():
    assert sha256_file(BUNDLE_ROOT / relative_path) == digest, f"Bundle checksum mismatch: {relative_path}"

csv_names = list(identity["csv_data_rows"])
tables = {Path(name).stem: pd.read_csv(BUNDLE_DIR / name) for name in csv_names}
for name, expected_rows in identity["csv_data_rows"].items():
    assert len(tables[Path(name).stem]) == expected_rows, f"Row-count mismatch: {name}"

required_columns = {
    "model_metrics": {"split", "ticker", "model_name", "paired_n", "mae", "rmse"},
    "pair_comparisons": {"comparison_id", "ticker", "paired_n", "delta_mae_baseline_minus_online"},
    "hypothesis_results": {"hypothesis_id", "ticker_or_macro", "dataset_split", "n_paired", "raw_p_value", "adjusted_p_value", "confidence_interval_95", "decision"},
    "coverage": {"split", "ticker", "model_name", "expected_eligible_prediction_anchors", "forecast_success_count", "mature_evaluable_count", "pending_label_count"},
    "evaluation": {"experiment_id", "prediction_date", "split", "ticker", "model_name", "predicted_vol_annualized", "target_realized_vol_5d", "absolute_error"},
    "forecast_audit": {"prediction_date", "split", "ticker", "model_name", "label_status", "forecast_status", "evaluation_status"},
    "drift_error_timeline": {"prediction_date", "ticker", "absolute_error_adwin", "drift_detected"},
    "portfolio_risk": {"split", "prediction_date", "model_name", "portfolio_vol_annualized", "portfolio_status", "covariance_matrix"},
}
for table_name, columns in required_columns.items():
    missing = columns - set(tables[table_name].columns)
    assert not missing, f"{table_name} is missing columns: {sorted(missing)}"

assert set(tables["evaluation"]["experiment_id"]) == {EXPECTED_REPLAY_ID}
assert set(tables["hypothesis_results"]["analysis_config_id"]) == {identity["analysis_config_id"]}
assert set(tables["evaluation"]["split"]) == {"test"}
assert set(tables["model_metrics"]["ticker"]) == {*TICKERS, "MACRO"}
assert set(tables["model_metrics"]["model_name"]) == set(MODEL_ORDER)

display(pd.DataFrame([
    {"Check": "Stage 06", "Result": manifest["status"]},
    {"Check": "Evaluation ID", "Result": identity["evaluation_id"]},
    {"Check": "Final Replay ID", "Result": identity["final_replay_experiment_id"]},
    {"Check": "Test window", "Result": "2025-10-01 to 2026-09-25"},
    {"Check": "Strict data cutoff", "Result": "2026-09-25"},
    {"Check": "Source status", "Result": identity["source_verification_status"]},
]))

## 02. Model overview

The names below are fixed by the project contract. These descriptions explain the role of each model; this notebook does not instantiate or fit any model.

In [ ]:
model_overview = pd.DataFrame([
    {"ID": "B0", "Model": "CONSTANT_BASELINE", "Role": "Per-ETF constant reference based on the Train target mean."},
    {"ID": "B1", "Model": "ROLLING_20D_BASELINE", "Role": "Rolling 20-session realized-volatility reference."},
    {"ID": "B2", "Model": "STATIC_SGD", "Role": "Train-fitted SGD model; scaler is Train-only and frozen."},
    {"ID": "B3", "Model": "ONLINE_SGD", "Role": "Starts from B2's initial state and selected ETF hyperparameters; updates only after labels mature, without duplicate updates."},
])
display(model_overview)

## 03. Test model performance

MAE and RMSE are annualized-decimal errors. The primary comparison is paired Macro MAE: SPY, IEF, and GLD receive equal weight. The portfolio's 50/30/20 weights are not used in this model comparison.

In [ ]:
metrics = tables["model_metrics"]
test_metrics = metrics.loc[metrics["split"].eq("test")].copy()
display(
    test_metrics.loc[:, ["ticker", "model_name", "paired_n", "expected_eligible_anchor_count", "mae", "rmse"]]
    .sort_values(["ticker", "model_name"])
    .reset_index(drop=True)
)

macro_metrics = test_metrics.loc[test_metrics["ticker"].eq("MACRO"), ["model_name", "paired_n", "mae", "rmse"]]
display(Markdown("**Paired Macro MAE:** each ETF has equal weight; paired N is 729 anchor pairs."))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
asset_metrics = test_metrics.loc[test_metrics["ticker"].isin(TICKERS)]
x = np.arange(len(TICKERS))
width = 0.19
for index, model in enumerate(MODEL_ORDER):
    values = [
        asset_metrics.loc[(asset_metrics["ticker"].eq(ticker)) & asset_metrics["model_name"].eq(model), "mae"].iloc[0]
        for ticker in TICKERS
    ]
    axes[0].bar(x + (index - 1.5) * width, values, width, label=MODEL_LABELS[model])
axes[0].set_xticks(x, TICKERS)
axes[0].set_ylabel("MAE (annualized decimal)")
axes[0].set_title("Test MAE by ETF")
axes[0].legend(title="Model")

macro_values = [
    macro_metrics.loc[macro_metrics["model_name"].eq(model), "mae"].iloc[0]
    for model in MODEL_ORDER
]
axes[1].bar([MODEL_LABELS[model] for model in MODEL_ORDER], macro_values, color="#315a8c")
axes[1].set_ylabel("Macro MAE (annualized decimal)")
axes[1].set_title("Equal-weight paired Macro MAE")
fig.suptitle("Model errors on the common B0–B3 Test panel")
fig.tight_layout()
fig.subplots_adjust(bottom=0.20)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | B0–B3 | "
    "N=243 per ETF; Macro N=729 paired ETF/date anchors | annualized decimal | "
    "common matured, valid B0–B3 panel; Macro gives each ETF equal weight."
)
fig.text(0.01, 0.01, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig01_model_mae_test.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "Test model MAE", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 04. Forecast and label coverage

A valid forecast may still have a Pending label at the end of the study window. Pending means there are not yet five future XNYS observations available for the label; it is not a forecast failure or an error score.

In [ ]:
audit = tables["forecast_audit"]
test_audit = audit.loc[audit["split"].eq("test")].copy()
test_expected = len(test_audit)
valid_forecasts = int(test_audit["forecast_status"].eq("VALID").sum())
failed_forecasts = test_expected - valid_forecasts
mature_evaluable = int(test_audit["evaluation_status"].eq("EVALUABLE").sum())
pending_labels = int(test_audit["evaluation_status"].eq("PENDING_LABEL").sum())
invalid_or_missing_targets = test_expected - mature_evaluable - pending_labels

assert test_expected == manifest["counts"]["test_predictions"] == 2976
assert mature_evaluable == manifest["counts"]["test_evaluable_predictions"] == 2916
assert pending_labels == manifest["counts"]["test_pending_label_rows"] == 60
assert failed_forecasts == manifest["counts"]["test_forecast_exceptions"] == 0

display(pd.DataFrame([
    {"Test audit state": "Expected prediction rows (744 ETF/date anchors × 4 models)", "Rows": test_expected},
    {"Test audit state": "Valid forecasts", "Rows": valid_forecasts},
    {"Test audit state": "Mature and evaluable", "Rows": mature_evaluable},
    {"Test audit state": "Pending labels", "Rows": pending_labels},
    {"Test audit state": "Invalid or missing target", "Rows": invalid_or_missing_targets},
    {"Test audit state": "Forecast failures", "Rows": failed_forecasts},
]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(["Expected", "Valid", "Failed"], [test_expected, valid_forecasts, failed_forecasts],
            color=["#8a9bab", "#3b8f63", "#c86464"])
axes[0].set_title("Forecast production")
axes[0].set_ylabel("Prediction rows")
axes[1].bar(["Mature / eval.", "Pending", "Invalid / missing"],
            [mature_evaluable, pending_labels, invalid_or_missing_targets],
            color=["#3b8f63", "#dfa844", "#c86464"])
axes[1].set_title("Target availability")
axes[1].set_ylabel("Prediction rows")
fig.suptitle("Test coverage: pending labels remain visible")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | N={test_expected} "
    "model/ETF/date rows from 744 eligible ETF/date anchors × 4 models | row counts | "
    "2,916 mature/evaluable; 60 pending; zero invalid/missing targets or forecast failures."
)
fig.text(0.01, 0.01, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig02_model_coverage.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "Test forecast and label coverage", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 05. Forecast inspection

This chart compares the recorded ONLINE_SGD forecasts with their realized Forward RV5 targets on the Test split. Only mature, evaluable labels are present in the evaluation table.

In [ ]:
evaluation = tables["evaluation"]
test_eval = evaluation.loc[evaluation["split"].eq("test")].copy()
online = test_eval.loc[test_eval["model_name"].eq("ONLINE_SGD")].copy()
assert online.groupby("ticker").size().to_dict() == {"GLD": 243, "IEF": 243, "SPY": 243}

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    series = online.loc[online["ticker"].eq(ticker)].sort_values("prediction_date")
    dates = pd.to_datetime(series["prediction_date"])
    ax.plot(dates, series["target_realized_vol_5d"],
            label="Actual Forward RV5", color="#262f3d", linewidth=1.3)
    ax.plot(dates, series["predicted_vol_annualized"],
            label="ONLINE_SGD forecast", color="#d06c3e", linewidth=1.1)
    ax.set_ylabel(f"{ticker}\nannualized decimal")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Prediction date")
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[-1].tick_params(axis="x", labelbottom=True)
plt.setp(axes[-1].get_xticklabels(), rotation=45, ha="right")
fig.suptitle("ONLINE_SGD forecast vs actual Forward RV5 — Test")
fig.tight_layout()
fig.subplots_adjust(bottom=0.18)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | model=ONLINE_SGD | "
    "N=243 per ETF, 729 total | annualized decimal | mature/evaluable rows only; "
    "actual is the next-five-session Forward RV5 target."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig03_forecast_vs_actual.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "ONLINE_SGD Forecast vs Actual", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 06. Temporal errors and ADWIN

Rolling MAE uses the approved 20-XNYS-session window and the Stage 06 minimum-evaluable rule. ADWIN is monitor-only. An alert is a signal to inspect errors, not proof that the underlying process has drifted.

In [ ]:
rolling = tables["rolling_mae20"]
online_rolling = rolling.loc[
    rolling["split"].eq("test") & rolling["model_name"].eq("ONLINE_SGD")
].copy()
drift = tables["drift_error_timeline"].copy()
alert_count = int(drift["drift_detected"].fillna(False).astype(bool).sum())
assert alert_count == 0
assert len(online_rolling) == 687

display(pd.DataFrame([
    {"Diagnostic": "ONLINE_SGD rolling windows", "Count": len(online_rolling)},
    {"Diagnostic": "Test common-panel error observations in drift timeline", "Count": len(drift)},
    {"Diagnostic": "ADWIN alerts", "Count": alert_count},
]))

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    roll = online_rolling.loc[online_rolling["ticker"].eq(ticker)].sort_values("window_end")
    alerts = drift.loc[
        drift["ticker"].eq(ticker) & drift["drift_detected"].fillna(False).astype(bool)
    ]
    ax.plot(pd.to_datetime(roll["window_end"]), roll["mae"], label="Rolling MAE20", color="#315a8c")
    if not alerts.empty:
        ax.scatter(pd.to_datetime(alerts["prediction_date"]), alerts["absolute_error_adwin"],
                   marker="x", color="#b82323", label="ADWIN alert")
    ax.set_ylabel(f"{ticker}\nMAE (decimal)")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Test session")
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[-1].get_xticklabels(), rotation=45, ha="right")
fig.suptitle("ONLINE_SGD rolling error and ADWIN monitor")
fig.tight_layout()
fig.subplots_adjust(bottom=0.18)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | model=ONLINE_SGD | "
    f"N={len(online_rolling)} rolling windows; drift timeline N={len(drift)} ETF/date rows | "
    "annualized decimal MAE; 20 XNYS sessions, minimum 10 evaluable | "
    f"ADWIN alerts={alert_count}; monitor-only, not proof of concept drift."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig04_rolling_mae20_and_adwin.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "Rolling MAE20 and ADWIN", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 07. H1 — Train-only association

H1 tests whether trailing five-session realized volatility is positively associated with Forward RV5. It uses Train only, OLS with Newey–West HAC lag 5, and Holm adjustment across the three ETFs. The 10- and 20-lag results are sensitivity checks. The notebook reads these results; it does not refit OLS or recompute p-values.

In [ ]:
h1 = tables["H1_results"].copy()
h1_sensitivity = tables["H1_sensitivity"].copy()
display(h1.loc[:, ["ticker", "n", "beta", "hac_se", "p_raw_one_sided", "p_holm",
                   "ci95_lower", "ci95_upper", "decision"]])
display(h1_sensitivity.loc[:, ["ticker", "hac_maxlags", "n", "beta",
                                "p_raw_one_sided", "ci95_lower", "ci95_upper"]])

scatter = tables["H1_scatter_data"]
train_scatter = scatter.loc[scatter["split"].eq("train")].copy()
assert len(train_scatter) == 2256
assert train_scatter.groupby("ticker").size().to_dict() == {"GLD": 752, "IEF": 752, "SPY": 752}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, ticker in zip(axes, TICKERS):
    points = train_scatter.loc[train_scatter["ticker"].eq(ticker)]
    ax.scatter(points["rv_5d"], points["target_realized_vol_5d"],
               s=11, alpha=0.45, color="#315a8c")
    ax.set_title(f"{ticker} (N={len(points)})")
    ax.set_xlabel("Trailing rv_5d")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Forward RV5")
fig.suptitle("Train-only trailing RV5 vs Forward RV5")
fig.tight_layout()
fig.subplots_adjust(bottom=0.20)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=train | date window=2021-09-27 to 2024-09-23 (eligible anchors) | ETF=SPY/IEF/GLD | "
    "N=752 per ETF | annualized decimal | H1 scatter data; association is not causation "
    "and does not establish Test forecast superiority."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig05_h1_train_scatter.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "H1 Train-only scatter", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 08. H2 and H3 — Paired Test comparisons

H2 compares ONLINE_SGD with STATIC_SGD; H3 compares ONLINE_SGD with ROLLING_20D_BASELINE. Both use paired Test outcomes on the same ETF/date anchors, a synchronized moving-block bootstrap of 10 XNYS sessions with 2,000 replicates, and Holm adjustment across the two tests.

Effects and confidence intervals below are read from Stage 06. The tests are one-sided; the displayed 95% intervals are two-sided. Neither interval is a probability that the null hypothesis is true.

In [ ]:
hypotheses = tables["hypothesis_results"]
h23 = hypotheses.loc[hypotheses["hypothesis_id"].isin(["H2", "H3"])].copy()
display(h23.loc[:, ["hypothesis_id", "dataset_split", "n_expected", "n_paired",
                    "method", "effect_size", "raw_p_value", "adjusted_p_value",
                    "confidence_interval_95", "decision"]])

effects = h23["effect_size"].astype(float).to_numpy()
intervals = np.array([json.loads(value) for value in h23["confidence_interval_95"]], dtype=float)
lower_error = effects - intervals[:, 0]
upper_error = intervals[:, 1] - effects
fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(h23))
ax.errorbar(effects, y, xerr=np.vstack([lower_error, upper_error]), fmt="o",
            capsize=5, color="#315a8c")
ax.axvline(0, color="#555555", linewidth=1, linestyle="--")
ax.set_yticks(y, h23["hypothesis_id"])
ax.set_xlabel("Macro DeltaMAE: baseline error − ONLINE_SGD error (annualized decimal)")
ax.set_title("Paired Test effects with Stage 06 two-sided 95% intervals")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.subplots_adjust(bottom=0.20)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | H2/H3 | N=729 paired ETF/date anchors | "
    "annualized decimal | Holm-adjusted p: H2=0.001999, H3=0.005997; two-sided 95% CIs copied from Stage 06; "
    "one-sided tests, Holm family contains H2 and H3."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig06_h2_h3_effect_ci.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "H2 and H3 paired effects", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 09. High-volatility subset

The high-volatility threshold is the 80th percentile of eligible Train Forward RV5, calculated separately for each ETF. Test rows above that frozen threshold form the subset. If an ETF has no qualifying Test rows, the result is shown as insufficient evidence instead of inventing a score.

In [ ]:
high_vol = tables["high_volatility_metrics"]
display(high_vol.loc[:, ["ticker", "model_name", "train_target_p80_threshold",
                         "threshold_train_n", "high_vol_common_evaluable_n", "mae", "rmse"]]
       .sort_values(["ticker", "model_name"]))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, ticker in zip(axes, TICKERS):
    rows = high_vol.loc[high_vol["ticker"].eq(ticker)].set_index("model_name").reindex(MODEL_ORDER)
    n = int(rows["high_vol_common_evaluable_n"].max())
    threshold = float(rows["train_target_p80_threshold"].iloc[0])
    ax.set_title(f"{ticker} | n={n}\nTrain P80={threshold:.4f}")
    if n == 0:
        ax.text(0.5, 0.5, "Insufficient evidence\n0 Test anchors above Train P80",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
    else:
        ax.bar([MODEL_LABELS[model] for model in MODEL_ORDER], rows["mae"].to_numpy(),
               color=["#8998a8", "#7691ad", "#527ba4", "#d06c3e"])
    ax.set_xlabel("Model")
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Test MAE (annualized decimal)")
fig.suptitle("Test error when Forward RV5 exceeds the ETF-specific Train P80")
fig.tight_layout()
fig.subplots_adjust(bottom=0.20)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | B0–B3 | "
    "threshold=eligible Train Forward RV5 80th percentile, per ETF | "
    "SPY N=15/model; IEF N=0/model (insufficient evidence); GLD N=189/model | "
    "annualized decimal MAE; no Test-derived threshold."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig07_high_vol_metrics.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "High volatility test metrics", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 10. Portfolio risk diagnostic

The Stage 06 portfolio series uses forecast volatility, 20-session correlations, and the default long-only weights SPY 50%, IEF 30%, GLD 20%. It is a risk diagnostic, not a portfolio return or investment-performance result.

The scenario table below is a consumer-side what-if calculation using the same stored covariance matrices and the approved formula. It does not retrain models and is not part of the Stage 06 model comparison.

In [ ]:
portfolio = tables["portfolio_risk"]
portfolio_test = portfolio.loc[
    portfolio["split"].eq("test") & portfolio["model_name"].eq("ONLINE_SGD")
].copy()
portfolio_test = portfolio_test.loc[portfolio_test["portfolio_status"].eq("OK")].copy()
assert len(portfolio_test) == 248
assert portfolio_test["portfolio_vol_annualized"].notna().all()

scenario_weights = {
    "Default 50/30/20": np.array([0.50, 0.30, 0.20]),
    "Equal 1/3 each": np.array([1 / 3, 1 / 3, 1 / 3]),
    "60/20/20": np.array([0.60, 0.20, 0.20]),
}
covariance_values = [
    np.asarray(json.loads(value), dtype=float)
    for value in portfolio_test["covariance_matrix"]
]
scenario_summary = []
for scenario_name, weights in scenario_weights.items():
    assert np.all(weights >= 0) and np.isclose(weights.sum(), 1.0)
    vols = np.array([np.sqrt(max(0.0, float(weights @ cov @ weights)))
                     for cov in covariance_values])
    if scenario_name == "Default 50/30/20":
        assert np.allclose(vols, portfolio_test["portfolio_vol_annualized"], atol=1e-10, rtol=0)
    scenario_summary.append({
        "Scenario": scenario_name,
        "Weights (SPY / IEF / GLD)": " / ".join(f"{weight:.3f}" for weight in weights),
        "Test dates": len(vols),
        "Mean forecast volatility": float(np.mean(vols)),
        "Median forecast volatility": float(np.median(vols)),
        "95th percentile": float(np.quantile(vols, 0.95)),
        "Status": "Derived what-if; same Stage 06 covariance inputs",
    })
scenario_table = pd.DataFrame(scenario_summary)
display(scenario_table)

fig, ax = plt.subplots(figsize=(12, 4.5))
series = portfolio_test.sort_values("prediction_date")
dates = pd.to_datetime(series["prediction_date"])
ax.plot(dates, series["portfolio_vol_annualized"],
        color="#315a8c", linewidth=1.4)
ax.set_xlabel("Prediction date")
ax.set_ylabel("Forecast portfolio volatility (annualized decimal)")
ax.set_title("ONLINE_SGD forecast-based portfolio risk — Test, default 50/30/20")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_xlim(pd.Timestamp("2025-10-01"), pd.Timestamp("2026-09-25"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.subplots_adjust(bottom=0.25)
caption = (
    f"Evaluation {identity['evaluation_id']} | split=test | date window=2025-10-01 to 2026-09-25 | model=ONLINE_SGD | "
    f"N={len(series)} complete-input dates | annualized decimal | "
    "20-session correlation proxy, default long-only weights 50/30/20; "
    "diagnostic only, not returns or portfolio performance."
)
fig.text(0.01, 0.005, caption, ha="left", va="bottom", fontsize=8, wrap=True)
figure_path = FIGURE_DIR / "fig08_portfolio_risk.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight",
            metadata={"Title": "Portfolio forecast risk", "Description": caption})
display_figure(figure_path, caption)
plt.close(fig)

## 11. Interpretation

- In this frozen Test snapshot, ONLINE_SGD has the lowest paired Macro MAE among B0–B3. This describes the approved historical replay, not guaranteed future performance.
- H2 and H3 reject their null hypotheses after the Stage 06 Holm adjustment. Report the paired N, effect, adjusted p-value, confidence interval, and one-sided test direction together.
- H1 shows positive Train-only association for each ETF. It is not evidence of causality or proof that a model will forecast well.
- All 2,976 expected Test forecasts are valid. Sixty labels remain Pending because the five-session outcome is not mature by the cutoff; 2,916 rows are evaluable.
- No ADWIN alerts were recorded. The monitor-only result does not establish that no future drift will occur.
- IEF has no Test anchors above its Train P80 threshold, so its high-volatility score is insufficient evidence.
- Portfolio volatility is forecast-based and uses a correlation proxy; it is not a return series or investment-performance claim.

## 12. Provenance and limitations

The approved snapshot is one Stage 06 evaluation linked to one completed Stage 05 Final Replay. The bundle's source identity, redacted manifests, original source hashes, exported-file hashes, and CHECKSUMS.sha256 are the provenance record.

**Interpretation status:** REVIEW READY. REPORT APPROVED still requires project-owner review.

**Known limitations**

- This is Simulated Historical Replay on a snapshot downloaded after the observation period. Historical point-in-time source availability is not verified.
- The results characterize one frozen snapshot and replay protocol; they do not establish prospective live performance or causality.
- H1_manifest records that the H1 analysis working tree was dirty when it ran. Its copied outputs and hashes are verified, but the precise uncommitted H1 code state is not archived here.
- ADWIN is monitor-only; no alerts in this run are not a guarantee against drift.
- The portfolio series is a forecast-based diagnostic with 20-session correlation inputs, not actual investment returns.
- High-volatility thresholds are derived from Train only. IEF has zero qualifying Test anchors.
- Statistical p-values, intervals, and decisions are read from Stage 06 and are not recomputed in this notebook.